
# S6E8 — Scientific Fork for Submission 8

**Goal:** reproduce the *style of analysis* used by the top notebook, then turn the strongest verified findings into a leak-safe competitive model.

This notebook asks:

1. Which features carry marginal signal?
2. Which features retain signal after conditioning?
3. Does the dataset contain a measurable generation constraint?
4. Can that constraint produce a useful residual feature?
5. Does missingness explain train/test drift?
6. Can leak-free value-level target encoding exploit the rounded value grid?
7. Does higher LightGBM capacity help?
8. Does the new model improve over our current **0.96553 leaderboard benchmark**?

This is inspired by the supplied Georgy Mamarin notebook; it is **not claimed to be an exact copy** of its hidden parameters or final submission. The supplied notebook reports a strong value-level target-encoding/high-capacity reference around 0.96668 OOF and 0.96740 after its `slack` feature, while its `other_screen` residual added about 0.00064 to its full model. fileciteturn1file0L90-L141



## 0. Competition benchmark

Our current best submission is:

**Submission 7 — 0.96553 LB**

So every experiment should be compared against:

- **Previous LB:** 0.96553
- **Previous approach:** 50% LightGBM + 20% CatBoost + 30% XGBoost
- **CV protocol:** 5-fold stratified CV

A small CV improvement is not automatically a real leaderboard improvement.


In [ ]:

import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/input/competitions/playground-series-s6e8"
TARGET = "addicted_label"
ID_COL = "id"

SEED = 42
N_SPLITS = 5
CURRENT_BEST_LB = 0.96553

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print("Train:", train.shape)
print("Test :", test.shape)
print("Target rate:", train[TARGET].mean())


## 1. Data contract and feature audit

In [ ]:

feature_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]

for c in feature_cols:
    print(
        f"{c:35s} dtype={str(train[c].dtype):8s} "
        f"missing={train[c].isna().mean():.2%} "
        f"nunique={train[c].nunique(dropna=True)}"
    )

print("\nTarget distribution:")
print(train[TARGET].value_counts(normalize=True).sort_index())



## 2. Signal atlas

The reference notebook starts with single-feature AUC, but warns that marginal AUC can be misleading. We reproduce that diagnostic first.

It found the three screen-time variables strongest, while `gaming_hours` and `work_study_hours` looked useful marginally but became approximately chance-level after conditioning. fileciteturn0file0L75-L96 fileciteturn0file0L138-L173


In [ ]:

def signed_auc(y_true, x):
    mask = pd.notna(x)
    if mask.sum() == 0 or y_true[mask].nunique() < 2:
        return np.nan
    return roc_auc_score(y_true[mask], x[mask])

signal_rows = []

for c in feature_cols:
    if pd.api.types.is_numeric_dtype(train[c]):
        auc = signed_auc(train[TARGET], train[c])
        auc_sym = max(auc, 1 - auc) if pd.notna(auc) else np.nan
    else:
        tmp = pd.DataFrame({"x": train[c].astype("object"), "y": train[TARGET]})
        vals = []
        for v in tmp["x"].dropna().unique():
            vals.append(roc_auc_score(tmp["y"], (tmp["x"] == v).astype(int)))
        auc_sym = max([max(a, 1-a) for a in vals], default=np.nan)
        auc = np.nan

    signal_rows.append({
        "feature": c,
        "single_feature_auc": auc,
        "symmetrised_auc": auc_sym,
        "missing_pct": train[c].isna().mean(),
        "nunique": train[c].nunique(dropna=True)
    })

signal_atlas = pd.DataFrame(signal_rows).sort_values(
    "symmetrised_auc", ascending=False
)
signal_atlas



## 3. Conditional AUC control

The key scientific question is:

> Given the strongest columns, does this feature still carry independent signal?

The reference notebook fixes daily screen time and social-media hours in small cells, then measures remaining features. Gaming averaged about 0.499 across those cells. fileciteturn0file0L100-L173


In [ ]:

DST = "daily_screen_time_hours"
SM = "social_media_hours"

conditional_cols = [
    c for c in [
        "gaming_hours",
        "work_study_hours",
        "sleep_hours",
        "app_opens_per_day"
    ] if c in train.columns
]

obs = train[[DST, SM] + conditional_cols + [TARGET]].dropna()
cells_out = []

for lo_d in [4, 5, 6, 7]:
    for lo_s, hi_s in [(1.0, 1.5), (1.5, 2.0), (2.0, 2.5)]:
        c = obs[
            (obs[DST] > lo_d) & (obs[DST] <= lo_d + 1) &
            (obs[SM] > lo_s) & (obs[SM] <= hi_s)
        ]

        if len(c) < 500 or c[TARGET].nunique() < 2:
            continue

        row = {
            "daily_cell": f"({lo_d}, {lo_d+1}]",
            "social_cell": f"({lo_s}, {hi_s}]",
            "n": len(c),
            "target_rate": c[TARGET].mean()
        }

        for col in conditional_cols:
            row[col] = roc_auc_score(c[TARGET], c[col])

        cells_out.append(row)

conditional_table = pd.DataFrame(cells_out)

print("Cells:", len(conditional_table))
print("\nMean signed AUC:")
print(conditional_table[conditional_cols].mean().round(4))
print("\nCells below 0.50:")
print((conditional_table[conditional_cols] < 0.50).sum())

conditional_table



## 4. Find the generation fingerprint

The reference notebook found:

`daily_screen_time_hours >= social_media_hours + gaming_hours + work_study_hours`

with zero violations in complete rows, then constructed the residual `other_screen`. fileciteturn0file0L377-L394


In [ ]:

constraint_cols = [
    DST,
    "social_media_hours",
    "gaming_hours",
    "work_study_hours"
]

complete = train[constraint_cols].dropna()

slack = (
    complete[DST]
    - complete["social_media_hours"]
    - complete["gaming_hours"]
    - complete["work_study_hours"]
)

print("Complete rows:", len(complete))
print("Violations:", int((slack < -1e-9).sum()))
print("Median slack:", slack.median())
print("Minimum slack:", slack.min())

train["other_screen"] = np.nan
test["other_screen"] = np.nan

mask_train = train[constraint_cols].notna().all(axis=1)
mask_test = test[constraint_cols].notna().all(axis=1)

train.loc[mask_train, "other_screen"] = (
    train.loc[mask_train, DST]
    - train.loc[mask_train, "social_media_hours"]
    - train.loc[mask_train, "gaming_hours"]
    - train.loc[mask_train, "work_study_hours"]
)

test.loc[mask_test, "other_screen"] = (
    test.loc[mask_test, DST]
    - test.loc[mask_test, "social_media_hours"]
    - test.loc[mask_test, "gaming_hours"]
    - test.loc[mask_test, "work_study_hours"]
)

resid_mask = train["other_screen"].notna()

print("\nCoverage:", resid_mask.mean())
print("Negative residuals:", int((train.loc[resid_mask, "other_screen"] < 0).sum()))
print(
    "Standalone residual AUC:",
    roc_auc_score(
        train.loc[resid_mask, TARGET],
        train.loc[resid_mask, "other_screen"]
    )
)



### Why `other_screen` matters

The reference notebook reported about 61% coverage, standalone AUC ≈ 0.7649, and a mean +0.00064 OOF gain when the residual was added to its full model. fileciteturn2file7L337-L352

We still measure it ourselves rather than assuming the same gain transfers to our setup.


## 5. Missingness and train/test drift

In [ ]:

missing_rows = []

for c in feature_cols:
    miss = train[c].isna()
    if miss.nunique() < 2:
        continue

    missing_rows.append({
        "feature": c,
        "missing_pct": miss.mean(),
        "target_if_missing": train.loc[miss, TARGET].mean(),
        "target_if_present": train.loc[~miss, TARGET].mean(),
        "gap": train.loc[miss, TARGET].mean() - train.loc[~miss, TARGET].mean()
    })

missing_target = pd.DataFrame(missing_rows).sort_values(
    "gap", key=lambda s: s.abs(), ascending=False
)

missing_target


In [ ]:

adv_train = train[feature_cols].isna().astype(np.int8)
adv_test = test[feature_cols].isna().astype(np.int8)

adv_X = pd.concat([adv_train, adv_test], ignore_index=True)
adv_y = np.r_[np.zeros(len(adv_train)), np.ones(len(adv_test))]

adv_oof = np.zeros(len(adv_X))
skf_adv = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)

for tr_i, va_i in skf_adv.split(adv_X, adv_y):
    model = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=SEED,
        verbosity=-1
    )
    model.fit(adv_X.iloc[tr_i], adv_y[tr_i])
    adv_oof[va_i] = model.predict_proba(adv_X.iloc[va_i])[:, 1]

print(
    "Adversarial AUC from missingness only:",
    roc_auc_score(adv_y, adv_oof)
)



## 6. Leak-free value-level target encoding

The reference notebook's strongest modeling lesson is **not** "use target encoding blindly."

It uses out-of-fold target encoding, with an inner split for training rows so a row never receives an encoding calculated from its own label. It then combines that with higher LightGBM capacity. fileciteturn3file0L90-L141

The default below is conservative. Expand `TE_COLS` only after measuring the gain.


In [ ]:

TE_COLS = [
    c for c in [
        "daily_screen_time_hours",
        "social_media_hours",
        "gaming_hours",
        "work_study_hours",
        "sleep_hours",
        "weekend_screen_time",
        "notifications_per_day",
        "app_opens_per_day",
        "age"
    ] if c in train.columns
]

INNER_FOLDS = 3
SMOOTH = 20.0

print("TE columns:", TE_COLS)


In [ ]:

def value_te_fold(tr_df, va_df, y_tr, cols, inner_folds=INNER_FOLDS, smooth=SMOOTH):
    # Leak-safe value-level target encoding.
    # Validation rows use the complete outer-training mapping.
    # Outer-training rows use inner OOF mappings.

    prior = float(np.mean(y_tr))
    A = pd.DataFrame(index=tr_df.index)
    B = pd.DataFrame(index=va_df.index)

    inner = list(
        StratifiedKFold(
            inner_folds,
            shuffle=True,
            random_state=SEED
        ).split(tr_df, y_tr)
    )

    for c in cols:
        kt = tr_df[c].astype("object")
        kv = va_df[c].astype("object")

        g = pd.DataFrame({
            "k": kt.to_numpy(),
            "y": np.asarray(y_tr)
        }).groupby("k")["y"].agg(["sum", "count"])

        full_map = (g["sum"] + prior * smooth) / (g["count"] + smooth)

        B["te_" + c] = (
            kv.map(full_map)
              .astype(float)
              .fillna(prior)
              .to_numpy()
        )

        col = np.full(len(tr_df), prior, dtype=float)

        for a, b in inner:
            gi = pd.DataFrame({
                "k": kt.iloc[a].to_numpy(),
                "y": np.asarray(y_tr)[a]
            }).groupby("k")["y"].agg(["sum", "count"])

            inner_map = (gi["sum"] + prior * smooth) / (gi["count"] + smooth)

            col[b] = (
                kt.iloc[b]
                .map(inner_map)
                .astype(float)
                .fillna(prior)
                .to_numpy()
            )

        A["te_" + c] = col

    return A, B


## 7. Build model matrices

In [ ]:

CAT_COLS = [
    c for c in ["gender", "stress_level", "academic_work_impact"]
    if c in train.columns
]

model_train = train[feature_cols + ["other_screen"]].copy()
model_test = test[feature_cols + ["other_screen"]].copy()

# Use category dtype so LightGBM receives true categorical features.
for c in CAT_COLS:
    categories = pd.Index(
        pd.concat([model_train[c], model_test[c]], axis=0)
        .astype("object")
        .dropna()
        .unique()
    )

    model_train[c] = pd.Categorical(model_train[c], categories=categories)
    model_test[c] = pd.Categorical(model_test[c], categories=categories)

print("Categoricals:", CAT_COLS)



## 8. High-capacity LightGBM candidate

The supplied notebook explicitly compares a weak 63-leaf/400-round reference with a stronger 255-leaf/1500-round reference plus value-level encoding. fileciteturn3file0L90-L139

This is our main Submission 8 candidate.


In [ ]:

def run_lgb_cv(use_residual=True, use_te=True, num_leaves=255, rounds=1500):
    X_base = model_train.copy()
    X_test_base = model_test.copy()

    feature_names = list(feature_cols)
    if use_residual:
        feature_names.append("other_screen")

    oof = np.zeros(len(train), dtype=float)
    pred = np.zeros(len(test), dtype=float)
    fold_scores = []

    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)

    for fold, (tr_i, va_i) in enumerate(skf.split(X_base, train[TARGET]), 1):
        X_tr = X_base.iloc[tr_i][feature_names].copy()
        X_va = X_base.iloc[va_i][feature_names].copy()
        X_te = X_test_base[feature_names].copy()

        y_tr = train[TARGET].iloc[tr_i].to_numpy()
        y_va = train[TARGET].iloc[va_i].to_numpy()

        if use_te:
            te_tr, te_va = value_te_fold(
                train.iloc[tr_i],
                train.iloc[va_i],
                y_tr,
                TE_COLS
            )

            prior = y_tr.mean()
            te_test = pd.DataFrame(index=np.arange(len(test)))

            for c in TE_COLS:
                kt = train.iloc[tr_i][c].astype("object")
                kv = test[c].astype("object")

                g = pd.DataFrame({
                    "k": kt.to_numpy(),
                    "y": y_tr
                }).groupby("k")["y"].agg(["sum", "count"])

                full_map = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)

                te_test["te_" + c] = (
                    kv.map(full_map)
                      .astype(float)
                      .fillna(prior)
                      .to_numpy()
                )

            for c in TE_COLS:
                X_tr["te_" + c] = te_tr["te_" + c"].to_numpy()
                X_va["te_" + c] = te_va["te_" + c].to_numpy()
                X_te["te_" + c] = te_test["te_" + c].to_numpy()

        cat_cols = [c for c in CAT_COLS if c in X_tr.columns]

        train_set = lgb.Dataset(
            X_tr,
            label=y_tr,
            categorical_feature=cat_cols,
            free_raw_data=False
        )
        val_set = lgb.Dataset(
            X_va,
            label=y_va,
            categorical_feature=cat_cols,
            reference=train_set,
            free_raw_data=False
        )

        params = {
            "objective": "binary",
            "metric": "auc",
            "learning_rate": 0.03,
            "num_leaves": num_leaves,
            "max_depth": -1,
            "min_child_samples": 30,
            "feature_fraction": 0.85,
            "bagging_fraction": 0.85,
            "bagging_freq": 1,
            "lambda_l1": 0.1,
            "lambda_l2": 0.1,
            "verbosity": -1,
            "seed": SEED + fold,
            "feature_pre_filter": False
        }

        model = lgb.train(
            params,
            train_set,
            num_boost_round=rounds,
            valid_sets=[val_set],
            callbacks=[
                lgb.early_stopping(100, verbose=False),
                lgb.log_evaluation(0)
            ]
        )

        val_pred = model.predict(X_va, num_iteration=model.best_iteration)
        test_pred = model.predict(X_te, num_iteration=model.best_iteration)

        oof[va_i] = val_pred
        pred += test_pred / N_SPLITS

        auc = roc_auc_score(y_va, val_pred)
        fold_scores.append(auc)

        print(
            f"Fold {fold}/{N_SPLITS} | "
            f"AUC={auc:.6f} | best_iter={model.best_iteration}"
        )

    print("\nOOF AUC:", roc_auc_score(train[TARGET], oof))
    print("Fold mean:", np.mean(fold_scores))
    print("Fold std :", np.std(fold_scores))

    return oof, pred, fold_scores



## 9. Run Submission 8

Start with:

- 13 original features
- `other_screen`
- leak-free value-level target encoding
- high-capacity LightGBM
- 5-fold stratified CV

If runtime is too high on Kaggle CPU, first reduce `TE_COLS` to the three screen-time variables.


In [ ]:

oof_s8, pred_s8, fold_scores_s8 = run_lgb_cv(
    use_residual=True,
    use_te=True,
    num_leaves=255,
    rounds=1500
)

s8_oof_auc = roc_auc_score(train[TARGET], oof_s8)

print("\n" + "=" * 70)
print(f"Submission 8 OOF AUC: {s8_oof_auc:.6f}")
print(f"Previous best LB     : {CURRENT_BEST_LB:.5f}")
print("=" * 70)



## 10. Optional paired ablation: does `other_screen` actually help?

Do not assume the reference notebook's gain transfers.

Run the comparison only after the main candidate completes. The reference notebook's residual gain was measured against the same folds and moved in the same direction across three seeds. fileciteturn2file7L326-L352


In [ ]:

# Optional; can take several minutes.
#
# oof_no_resid, pred_no_resid, _ = run_lgb_cv(
#     use_residual=False,
#     use_te=True,
#     num_leaves=255,
#     rounds=1500
# )
#
# auc_with = roc_auc_score(train[TARGET], oof_s8)
# auc_without = roc_auc_score(train[TARGET], oof_no_resid)
#
# print("With residual   :", auc_with)
# print("Without residual:", auc_without)
# print("Residual gain   :", auc_with - auc_without)



## 11. Submission generation

The reference notebook validates row count, ID alignment, prediction range, and missing predictions before writing the submission. fileciteturn3file0L55-L74


In [ ]:

submission8 = sample_submission.copy()
submission8[TARGET] = pred_s8

assert len(submission8) == len(test)
assert submission8[ID_COL].equals(test[ID_COL])
assert submission8[TARGET].notna().all()
assert submission8[TARGET].between(0, 1).all()

submission8.to_csv("submission8_scientific_fork.csv", index=False)
np.save("submission8_oof.npy", oof_s8)

print("Saved: submission8_scientific_fork.csv")
print("Rows :", len(submission8))
print("Mean :", submission8[TARGET].mean())
print("Range:", submission8[TARGET].min(), submission8[TARGET].max())
submission8.head()



# 12. Decision rule

### If Submission 8 improves OOF and the ablation is coherent
Submit it and compare the leaderboard against **0.96553**.

### If OOF improves but the gain is fragile
Test another seed/fold setup before trusting it.

### If Submission 8 is worse
Keep Submission 7 as the benchmark and move to the next experiment.

The reference notebook's central lesson is the same: diagnose → measure → ablate → model → submit, rather than treating every score increase as proof of a mechanism. fileciteturn0file0L177-L239

## What we deliberately are not doing yet

- No arbitrary ratio-feature zoo
- No monotonic constraint
- No blind XGBoost addition
- No target leakage
- No target encoding calculated from a row's own label
- No claim that a higher CV score guarantees a higher leaderboard score
- No treating the candidate source dataset as proven provenance
